In [1]:
# ==============================================================================
# 1. IMPORTAÇÕES NECESSÁRIAS
# ==============================================================================
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error


In [13]:
# ==============================================================================
# 2. FUNÇÕES OTIMIZADAS PARA TREINAMENTO E AVALIAÇÃO
# ==============================================================================

# As funções de passo de treino/validação são compiladas para máxima performance.
@tf.function
def train_step(model, x_batch, y_batch, loss_fn, optimizer):
    """Executa um único passo de treino em um lote de dados."""
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        loss = loss_fn(y_batch, predictions)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

@tf.function
def val_step(model, x_batch, y_batch, loss_fn):
    """Executa um único passo de validação em um lote de dados."""
    predictions = model(x_batch, training=False)
    val_loss = loss_fn(y_batch, predictions)
    return val_loss

def root_mean_squared_error_np(y_true, y_pred):
    """Calcula o RMSE usando NumPy para o relatório final."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def train_model_efficient(model,
                          optimizer,  # NOVO PARÂMETRO
                          loss_fn,    # NOVO PARÂMETRO
                          train_dataset, val_dataset, test_dataset,
                          epochs=5000, patience=5, min_delta=0, 
                          plot=True):
    """
    Função principal de treinamento que orquestra todo o processo otimizado.
    Recebe o otimizador e a loss_fn como argumentos para evitar recriação.
    """
    # As variáveis de controle e histórico são inicializadas aqui
    history = {'loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_weights = None

    # Otimizador e loss_fn agora são recebidos, não criados.
    print(f"Iniciando o treinamento (LR={optimizer.learning_rate.numpy():.4f}, Paciência={patience})...")
    
    # LOOP DE TREINAMENTO PRINCIPAL
    for epoch in range(epochs):
        # -- Loop de Treino sobre os mini-lotes --
        epoch_train_loss = 0.0
        num_batches = 0
        for x_batch, y_batch in train_dataset:
            batch_loss = train_step(model, x_batch, y_batch, loss_fn, optimizer)
            epoch_train_loss += batch_loss
            num_batches += 1
        avg_train_loss = epoch_train_loss / num_batches

        # -- Loop de Validação sobre os mini-lotes --
        epoch_val_loss = 0.0
        num_val_batches = 0
        for x_val_batch, y_val_batch in val_dataset:
            batch_val_loss = val_step(model, x_val_batch, y_val_batch, loss_fn)
            epoch_val_loss += batch_val_loss
            num_val_batches += 1
        avg_val_loss = epoch_val_loss / num_val_batches
        
        history['loss'].append(avg_train_loss.numpy())
        history['val_loss'].append(avg_val_loss.numpy())

        # LÓGICA DE EARLY STOPPING
        if avg_val_loss < best_val_loss - min_delta:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            best_weights = model.get_weights()
        else:
            epochs_no_improve += 1

        if (epoch + 1) % 100 == 0 or epoch == 0 or epochs_no_improve >= patience:
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.5f}, Val Loss: {avg_val_loss:.5f} (Melhora há {epochs_no_improve} épocas)")

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping na época {epoch+1}! Validação não melhorou por {patience} épocas.")
            print(f"Melhor Val Loss: {best_val_loss:.5f} (na época {epoch + 1 - epochs_no_improve})")
            if best_weights:
                model.set_weights(best_weights)
                print("Pesos do modelo restaurados para a melhor época.")
            break
            
    # =================================================
    # AVALIAÇÃO FINAL E PLOTS
    # =================================================
    print("\nCalculando métricas finais com o melhor modelo...")

    train_pred = model.predict(train_dataset)
    val_pred = model.predict(val_dataset)
    test_pred = model.predict(test_dataset)

    train_y_true = np.concatenate([y for x, y in train_dataset], axis=0)
    y_val_true = np.concatenate([y for x, y in val_dataset], axis=0)
    y_test_true = np.concatenate([y for x, y in test_dataset], axis=0)

    metrics = {
        'Training': {
            'R2': r2_score(train_y_true, train_pred),
            'MSE': mean_squared_error(train_y_true, train_pred),
            'RMSE': root_mean_squared_error_np(train_y_true, train_pred),
            'MAPE': mean_absolute_percentage_error(train_y_true, train_pred)
        },
        'Validation': {
            'R2': r2_score(y_val_true, val_pred),
            'MSE': mean_squared_error(y_val_true, val_pred),
            'RMSE': root_mean_squared_error_np(y_val_true, val_pred),
            'MAPE': mean_absolute_percentage_error(y_val_true, val_pred)
        },
        'Test': {
            'R2': r2_score(y_test_true, test_pred),
            'MSE': mean_squared_error(y_test_true, test_pred),
            'RMSE': root_mean_squared_error_np(y_test_true, test_pred),
            'MAPE': mean_absolute_percentage_error(y_test_true, test_pred)
        }
    }

    for name, m in metrics.items():
        print(f"\n{name} Metrics:")
        for k, v in m.items():
            print(f"  {k}: {v:.4f}")

    if plot:
        plt.figure(figsize=(8, 5))
        plt.plot(history['loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('MSE Loss')
        plt.title('Training History')
        plt.legend()
        plt.grid(True)
        plt.show()

        fig, axs = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
        datasets_plot = [
            ('Treinamento', train_y_true, train_pred), 
            ('Validação', y_val_true, val_pred),
            ('Teste', y_test_true, test_pred)
        ]

        for i, (title, y_true, y_pred) in enumerate(datasets_plot):
            axs[i].plot(y_true, 'o-', label='Amostras Reais', markersize=4, alpha=0.7)
            axs[i].plot(y_pred, 'x-', label='Valores Preditos', markersize=4, alpha=0.7)
            axs[i].set_title(title)
            axs[i].set_ylabel('Valor')
            axs[i].legend()
            axs[i].grid(True)
        
        axs[-1].set_xlabel('Índice da Amostra')
        plt.suptitle('Comparação: Amostras Reais vs. Valores Preditos', fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

    return metrics

In [15]:
# ==============================================================================
# 3. BLOCO DE EXECUÇÃO PRINCIPAL (MAIN)
# ==============================================================================

if __name__ == "__main__":
    # --- PARÂMETROS DE CONFIGURAÇÃO ---
    TIMESTEPS = 5
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    PREDICTORS = ["Wd", "We"]
    TARGET = "theta(Wd,We)"
    FEATURES = len(PREDICTORS)
    
    # --- CARREGAMENTO DOS DADOS A PARTIR DOS ARQUIVOS ORIGINAIS ---
    # Este bloco agora usa os caminhos dos seus arquivos CSV.
    # Certifique-se de que o diretório "./Dados/" existe no mesmo local do seu script.
    try:
        print("Carregando dados dos arquivos CSV...")
        TrainData = pd.read_csv("./Dados/DataSpline.csv").drop(columns=["x(Wd,We)", "y(Wd,We)"])
        ValData = pd.read_csv("./Dados/DataSpline3.csv").drop(columns=["x(Wd,We)", "y(Wd,We)"])
        TestData = pd.read_csv("./Dados/DataSpline2.csv").drop(columns=["x(Wd,We)", "y(Wd,We)"])
        print("Arquivos CSV carregados com sucesso.")
    except FileNotFoundError as e:
        print(f"Erro: Arquivo não encontrado - {e}")
        print("Verifique se os arquivos CSV estão no diretório './Dados/' relativo à localização do seu script.")
        exit() # Encerra o script se os dados não puderem ser carregados

    # --- PRÉ-PROCESSAMENTO E CRIAÇÃO DOS DATASETS ---
    train_x_raw = TrainData[PREDICTORS].to_numpy().astype(np.float32)
    train_y_raw = TrainData[TARGET].to_numpy().astype(np.float32)

    x_val_raw = ValData[PREDICTORS].to_numpy().astype(np.float32)
    y_val_raw = ValData[TARGET].to_numpy().astype(np.float32)

    x_test_raw = TestData[PREDICTORS].to_numpy().astype(np.float32)
    y_test_raw = TestData[TARGET].to_numpy().astype(np.float32)
    
    print("\nCriando pipelines de dados de séries temporais...")
    
    train_dataset = keras.utils.timeseries_dataset_from_array(
        data=train_x_raw,
        targets=train_y_raw[TIMESTEPS:],
        sequence_length=TIMESTEPS,
        batch_size=BATCH_SIZE,
        shuffle=True
    ).prefetch(tf.data.AUTOTUNE)

    val_dataset = keras.utils.timeseries_dataset_from_array(
        data=x_val_raw,
        targets=y_val_raw[TIMESTEPS:],
        sequence_length=TIMESTEPS,
        batch_size=BATCH_SIZE
    ).prefetch(tf.data.AUTOTUNE)

    test_dataset = keras.utils.timeseries_dataset_from_array(
        data=x_test_raw,
        targets=y_test_raw[TIMESTEPS:],
        sequence_length=TIMESTEPS,
        batch_size=BATCH_SIZE
    ).prefetch(tf.data.AUTOTUNE)

    # --- DEFINIÇÃO DO MODELO E TREINAMENTO ---
    model = keras.models.Sequential([
        keras.layers.SimpleRNN(20, return_sequences=False, input_shape=[TIMESTEPS, FEATURES]),
        keras.layers.Dense(1)
    ])
    model.summary()

    # Crie o otimizador e a função de perda aqui, uma única vez.
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    loss_fn = keras.losses.MeanSquaredError()

    # Executa o treinamento completo, passando os objetos como argumentos.
    final_metrics = train_model_efficient(
        model,
        optimizer,
        loss_fn,
        train_dataset,
        val_dataset,
        test_dataset,
        epochs=5000, 
        patience=5
    )

Carregando dados dos arquivos CSV...
Arquivos CSV carregados com sucesso.

Criando pipelines de dados de séries temporais...


c:\Users\João Vitor\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)        │ (None, 20)             │           460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481 (1.88 KB)

 Trainable params: 481 (1.88 KB)

 Non-trainable params: 0 (0.00 B)

Iniciando o treinamento (LR=0.0010, Paciência=5)...


ValueError: in user code:

    File "C:\Users\João Vitor\AppData\Local\Temp\ipykernel_7056\164694426.py", line 14, in train_step  *
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    File "c:\Users\João Vitor\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py", line 383, in apply_gradients  **
        self.apply(grads, trainable_variables)
    File "c:\Users\João Vitor\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py", line 422, in apply
        self.build(trainable_variables)
    File "c:\Users\João Vitor\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\adam.py", line 97, in build
        self.add_variable_from_reference(
    File "c:\Users\João Vitor\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\backend\tensorflow\optimizer.py", line 35, in add_variable_from_reference
        return super().add_variable_from_re